# Visual Concept Crystallization

**Analogue of:** `entity_processing.ipynb` (§4.3 of Patchscopes paper)

**Research question:** How does a VLM's understanding of an image patch evolve across LLM backbone layers?

For text, the paper showed that a token representing 'Diana, Princess of Wales' progresses
from `'Country in the United Kingdom'` (layer 1–2, model only sees 'Wales') to a fully
resolved description by layer 6. This notebook asks the same question visually:
at which layer does a patch token representing a dog's face become semantically
recognizable as such?

**Method:** Entity Description Patchscope  
- Source: image + text prompt; extract visual patch token hidden state at each layer  
- Target: `"Syria: Country in the Middle East, Leonardo DiCaprio: American actor, x:"`  
- Patch at position `x`; generate and score with RougeL vs. ground truth region description  

**Dataset needed:** Visual Genome region descriptions  
https://visualgenome.org/api/v0/api_endpoint_descriptions


In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from general_utils import ModelAndTokenizer
from patchscopes_utils import (
    set_hs_patch_hooks_llava_batch,
    inspect_vlm,
    evaluate_visual_entity_resolution_batch,
)

## 1. Load Model

In [ ]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

import torch
mt = ModelAndTokenizer(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device="cuda",
)
# Wire the batch hook setter so existing inspect/batch functions work
mt.set_hs_patch_hooks = set_hs_patch_hooks_llava_batch

print(mt)
print(f"is_vlm={mt.is_vlm}, num_layers={mt.num_layers}, num_visual_tokens={mt.num_visual_tokens}")

## 2. Load Dataset

Each row describes an image region with:
- `image_path`: path to the image file
- `patch_index`: which of the 576 visual patches (0–575) covers this region
- `reference_description`: ground truth description of the region (from Visual Genome)
- `popularity`: 'high' or 'low' (based on how often the concept appears in training data)

In [ ]:
# TODO: set path to your Visual Genome region description dataset
DATASET_PATH = "./preprocessed_data/visual_genome_regions.tsv"

dataset_df = pd.read_csv(DATASET_PATH, sep="\t")
print(f"Loaded {len(dataset_df)} samples")
dataset_df.head()

## 3. Build Experiment DataFrame

One row per (sample, layer_source, layer_target) combination.

For concept crystallization we use `layer_source == layer_target` (same layer),
matching the Token Identity Patchscope design from the paper (§4.1).

In [ ]:
# Entity description target prompt (same as text experiment §4.3)
TARGET_PROMPT = "Syria: Country in the Middle East, Leonardo DiCaprio: American actor, Samsung: South Korean multinational corporation, x:"

# Source text prompt for the VLM.
# NOTE: <image> must come first, immediately after BOS — get_hidden_state_vlm
# assumes mt.visual_token_start=1 (visual patches start right after BOS).
SOURCE_PROMPT = "<image>\nUSER: Describe this image. ASSISTANT:"

rows = []
for _, row in dataset_df.iterrows():
    for layer in range(mt.num_layers):
        rows.append({
            "image_path": row["image_path"],
            "prompt_source": SOURCE_PROMPT,
            "prompt_target": TARGET_PROMPT,
            "position_source": int(row["patch_index"]),
            "position_target": -1,
            "layer_source": layer,
            "layer_target": layer,  # same-layer patching
            "modality": "visual",
            "reference_description": row["reference_description"],
            "popularity": row.get("popularity", "unknown"),
        })

exp_df = pd.DataFrame(rows)
print(f"Experiment rows: {len(exp_df)} ({len(dataset_df)} samples × {mt.num_layers} layers)")

## 4. Run Evaluation

In [ ]:
results = evaluate_visual_entity_resolution_batch(
    mt,
    exp_df,
    batch_size=32,
    max_gen_len=30,
    transform=None,
)

exp_df["rouge_l"] = results["rouge_l"]
exp_df["rouge_1"] = results["rouge_1"]
exp_df["generation"] = results["generations"]

# Save results
exp_df.to_csv("./results_visual_concept_crystallization.csv", index=False)
print("Results saved.")

## 5. Visualize

RougeL vs. layer depth, split by popularity (high vs. low).

In [ ]:
mean_by_layer = exp_df.groupby(["layer_source", "popularity"])["rouge_l"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
for pop, grp in mean_by_layer.groupby("popularity"):
    ax.plot(grp["layer_source"], grp["rouge_l"], marker="o", label=pop)
ax.set_xlabel("LLM Backbone Layer")
ax.set_ylabel("RougeL (vs. region description)")
ax.set_title("Visual Concept Crystallization Across Layers")
ax.legend(title="Visual Concept Popularity")
plt.tight_layout()
plt.savefig("./visual_concept_crystallization.png", dpi=150)
plt.show()

In [ ]:
# Inspect a single example across layers
sample_image_path = dataset_df["image_path"].iloc[0]
sample_patch = int(dataset_df["patch_index"].iloc[0])
sample_image = Image.open(sample_image_path).convert("RGB")

print("Reference:", dataset_df["reference_description"].iloc[0])
print()
for layer in range(0, mt.num_layers, 4):
    gen = inspect_vlm(
        mt,
        image=sample_image,
        prompt_source=SOURCE_PROMPT,
        prompt_target=TARGET_PROMPT,
        layer_source=layer,
        layer_target=layer,
        position_source=sample_patch,
        position_target=-1,
        modality="visual",
        generation_mode=True,
        max_gen_len=30,
    )
    print(f"Layer {layer:2d}: {gen}")